# Randomization of wave 2 lab groups into control and treated groups

In [ ]:
# Create break to prevent this being re-run accidentally
raise Exception("This code has already been run. Do not run again to avoid overwriting existing files.")

In [2]:
# Set up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config

In [3]:
# Load datasets
labs = pd.read_excel(config.WAVE2_LABS_LIST / f"FinalVetsuisseLabs.xlsx")
existing_labs = pd.read_csv(config.WAVE1_LABS_LIST / "LabsList_Randomized.csv")
existing_labs_locations = pd.read_csv(config.WAVE1_LABS_LIST / "LabsList_Randomized_Locations.csv")

In [4]:
# Remove all rows where all values are missing
labs = labs.dropna(how="all")

In [5]:
# Count labs with an exclusion reason
count_exclusion_notice = labs["Exclusion Notice"].notna().sum()
print(f"Number of labs with an exclusion notice: {count_exclusion_notice}")

# Flag labs to exclude
labs["Exclude"] = labs["Exclusion Notice"].notna()
count_exclude = (labs["Exclude"] == 1).sum()
print(f"Number of labs to exclude: {count_exclude}")

# Remove these labs from the sample
labs_sample = labs[~labs["Exclude"]].copy()

Number of labs with an exclusion notice: 8
Number of labs to exclude: 8


In [6]:
# Clean the labs data

# Check that no missing lab groups names
labs_missing_name = labs_sample[labs_sample["Lab Group"].isna()]
assert len(labs_missing_name) == 0, "There are lab groups without a name. Please check the data."

# Check that no duplicate lab group names
duplicate_labs = labs_sample[labs_sample.duplicated(subset=["Lab Group"], keep=False)]
assert len(duplicate_labs) == 0, "There are duplicate lab group names. Please check the data."

# Check for duplicate professors (only if not missing)
duplicate_professors = labs_sample[labs_sample.duplicated(subset=["Professor"], keep=False) & labs_sample["Professor"].notna()]
if len(duplicate_professors) > 0:
    print("Warning: There are duplicate professors. Please check the data.")

# Check for missing lab group professors
labs_missing_prof = labs_sample[labs_sample["Professor"].isna()]
print(f"Lab groups without a prof: {len(labs_missing_prof)}")

# Check for missing lab group emails
labs_missing_email = labs_sample[labs_sample["Email"].isna()]
assert len(labs_missing_email) == 0, "There are lab groups without an email. Please check the data."

# Replace "VetSuisse" with "Vetsuisse" in the Faculty column
labs_sample["Faculty"] = labs_sample["Faculty"].replace("VetSuisse", "Vetsuisse")

# Check that faculty is "Vetsuisse" for all labs
labs_wrong_faculty = labs_sample[labs_sample["Faculty"] != "Vetsuisse"]
assert len(labs_wrong_faculty) == 0, "There are lab groups with a different faculty than Vetsuisse. Please check the data."

# Check for missing lab group websites
labs_missing_website = labs_sample[labs_sample["Source"].isna()]
print(f"Lab groups without a website: {len(labs_missing_website)}")

# Drop unnecessary columns
labs_sample = labs_sample.dropna(axis=1, how="all")
labs_sample = labs_sample.drop(columns=["Exclude"])

Lab groups without a prof: 2
Lab groups without a website: 0


In [7]:
# Create lab group id (random and secure) (exclude existing lab group ids)
np.random.seed(config.SEED)
existing_lab_ids = existing_labs["labgroupid"].unique()
n_labs = len(labs_sample)
possible_ids = np.arange(100, 999)
available_ids = np.setdiff1d(possible_ids, existing_lab_ids) # Available IDs excluding existing ones
labs_sample["labgroupid"] = np.random.choice(available_ids, size=n_labs, replace=False)

# Reorder columns
order = [
    "labgroupid", "Lab Group", "Faculty", "Institute", 
    "Professor", "Email", "Source"
]
new_order = [col for col in order if col in labs_sample.columns]
labs_sample = labs_sample[new_order]

In [ ]:
# Randomize wave 2 lab groups into treatment and control (50/50, stratified by faculty)
np.random.seed(2805) # date of running code in MMDD format
def stratified_randomize(df, group_col, treatment_col="Treatment Status"):
    """Randomly assign 50/50 treatment and control within each group,
       randomly assigning the extra lab if group size is odd."""
    def assign(group):
        n = len(group)
        labels = ["treatment"] * (n // 2) + ["control"] * (n // 2)
        if n % 2 == 1:  # randomly assign extra lab if odd
            labels.append(np.random.choice(["treatment", "control"]))
        np.random.shuffle(labels)
        group[treatment_col] = labels
        return group
    return df.groupby(group_col, group_keys=False).apply(assign)

labs_sample = stratified_randomize(labs_sample, group_col="Faculty")

# See how many treatment and control labs
print(labs_sample["Treatment Status"].value_counts())

# Save the wave 2 labs list
cols_to_save = [col for col in labs_sample.columns]
labs_sample.to_csv(config.WAVE2_LABS_LIST / f"LabsList_Randomized_Wave2.csv", index=False, columns=cols_to_save)

Treatment Status
treatment    9
control      9
Name: count, dtype: int64
Treatment Status  Faculty  
control           Vetsuisse    9
treatment         Vetsuisse    9
Name: count, dtype: int64


/var/folders/nn/21zflm3n7gzc5spw42wq352rpm87xt/T/ipykernel_4197/2792698486.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby(group_col, group_keys=False).apply(assign)


In [9]:
# Combine with existing randomized labs
all_randomized_labs = pd.concat([existing_labs_locations, labs_sample], ignore_index=True)

In [10]:
# Save the combined waves labs lists (with and without locations)

# Save the labs list (no locations)
cols_to_save = [col for col in all_randomized_labs.columns if col not in ["Location SCH", "Location BOT"]]
all_randomized_labs.to_csv(config.COMBINED_WAVES_LABS_LIST / f"LabsList_Randomized_Combined_Waves.csv", index=False, columns=cols_to_save)

# Save the labs list (with locations)
all_randomized_labs.to_csv(config.COMBINED_WAVES_LABS_LIST / f"LabsList_Randomized_Locations_Combined_Waves.csv", index=False)
